In [ ]:
import html
import json
import ipywidgets as widgets
from IPython.display import display


# Prototype governed-table catalogue.
# Production would populate this from FabricOps metadata.
TABLE_CATALOGUE = [
    {
        "environment": "dev",
        "store": "Silver",
        "store_type": "Lakehouse",
        "schema": "demo",
        "table_name": "curated_orders",
        "rows": 120,
        "columns": 9,
        "load_strategy": "overwrite",
        "profiled_at": "22 Sep 2026 20:10",
        "table_id": "73c7120a70369887049ed5f75aaa94af61cbf8a5abe6fdfb254c312a573a04d8",
    },
    {
        "environment": "dev",
        "store": "Bronze",
        "store_type": "Lakehouse",
        "schema": "demo",
        "table_name": "orders",
        "rows": 120,
        "columns": 9,
        "load_strategy": "append",
        "profiled_at": "22 Sep 2026 20:02",
        "table_id": "d1b4128652381586640a3bd111ea18bc8d7b12257ac91d36ed6a3f40b09f80dc",
    },
    {
        "environment": "dev",
        "store": "Gold",
        "store_type": "Warehouse",
        "schema": "demo",
        "table_name": "customer_summary",
        "rows": 60,
        "columns": 4,
        "load_strategy": "overwrite",
        "profiled_at": "22 Sep 2026 20:06",
        "table_id": "102bb4f01176587d0cdca46d546000000000000000000000000000000000000",
    },
]


state = {
    "table": {
        "environment": "dev",
        "store": "Silver",
        "store_type": "Lakehouse",
        "schema": "demo",
        "table_name": "curated_orders",
        "rows": 120,
        "columns": 9,
        "load_strategy": "overwrite",
        "profiled_at": "22 Sep 2026 20:10",
    },

    "contract": {
        "version": 1,
        "status": "DRAFT",
    },

    "table_definition": {
        "description": "Curated order level dataset used for downstream analytics.",
        "classification": "Internal",
        # Prototype display field only. Keep separate from Freshness guardrail semantics.
        "refresh_frequency": "Daily",

        "freshness_enabled": True,
        "freshness_column": "order_datetime",
        "freshness_age": 24,
        "freshness_unit": "Hours",
        "freshness_block": True,

        "source_drift_enabled": True,
        "source_drift_partition": "",
        "source_drift_change": "order_datetime",
        "source_drift_block": False,
    },

    "columns": [
        {
            "name": "order_id",
            "type": "string",
            "description": "Unique order identifier",
            "classification": "Internal",
            "required": True,
            "sensitive_enabled": False,
            "treatment": "",
            "sensitive_block": True,
            "dq": {
                "unique_values": {
                    "enabled": True,
                    "block": True,
                },
            },
        },
        {
            "name": "customer_id",
            "type": "string",
            "description": "Customer identifier used across orders",
            "classification": "Confidential",
            "required": True,
            "sensitive_enabled": True,
            "treatment": "tokenize",
            "sensitive_block": True,
            "dq": {
                "missing_values": {
                    "enabled": True,
                    "maximum_null_percent": 0.0,
                    "block": True,
                },
            },
        },
        {
            "name": "product_id",
            "type": "string",
            "description": "Product identifier",
            "classification": "Internal",
            "required": True,
            "sensitive_enabled": False,
            "treatment": "",
            "sensitive_block": True,
            "dq": {},
        },
        {
            "name": "order_datetime",
            "type": "timestamp",
            "description": "Date and time when the order was received",
            "classification": "Internal",
            "required": True,
            "sensitive_enabled": False,
            "treatment": "",
            "sensitive_block": True,
            "dq": {},
        },
        {
            "name": "quantity",
            "type": "integer",
            "description": "",
            "classification": "",
            "required": False,
            "sensitive_enabled": False,
            "treatment": "",
            "sensitive_block": True,
            "dq": {},
        },
        {
            "name": "unit_price",
            "type": "decimal",
            "description": "",
            "classification": "",
            "required": False,
            "sensitive_enabled": False,
            "treatment": "",
            "sensitive_block": True,
            "dq": {},
        },
        {
            "name": "order_net_amount",
            "type": "decimal",
            "description": "",
            "classification": "",
            "required": False,
            "sensitive_enabled": False,
            "treatment": "",
            "sensitive_block": True,
            "dq": {},
        },
        {
            "name": "historical_net_amount",
            "type": "decimal",
            "description": "",
            "classification": "",
            "required": False,
            "sensitive_enabled": False,
            "treatment": "",
            "sensitive_block": True,
            "dq": {},
        },
        {
            "name": "status",
            "type": "string",
            "description": "",
            "classification": "",
            "required": False,
            "sensitive_enabled": False,
            "treatment": "",
            "sensitive_block": True,
            "dq": {
                "accepted_values": {
                    "enabled": True,
                    "values": "Pending, Completed, Cancelled",
                    "block": False,
                },
            },
        },
    ],

    "relationships": {
        "composite_unique": [
            {
                "columns": ["order_id", "product_id"],
                "block": True,
            },
        ],
        "compare_columns": [],
        "required_when": [],
        "conditional_value": [],
        "referential_columns": [],
    },
}


COLUMN_RULES = {
    "missing_values": {
        "title": "Missing values",
        "description": "Controls how much missing data is allowed for this column.",
        "types": "all",
    },
    "blank_text": {
        "title": "Blank text",
        "description": "Rejects blank text values while allowing non-blank strings.",
        "types": {"string"},
    },
    "unique_values": {
        "title": "Unique values",
        "description": "Requires every non-null value in this column to be unique.",
        "types": "all",
    },
    "accepted_values": {
        "title": "Accepted values",
        "description": "Restricts this column to an approved set of values.",
        "types": "all",
    },
    "blocked_values": {
        "title": "Blocked values",
        "description": "Rejects a specified set of values for this column.",
        "types": "all",
    },
    "value_range": {
        "title": "Value range",
        "description": "Checks whether values stay within an expected range.",
        "types": {"integer", "decimal", "timestamp"},
    },
    "text_pattern": {
        "title": "Pattern match",
        "description": "Checks whether text values match a pattern.",
        "types": {"string"},
    },
}




# Prototype profile context only.
# In the real widget this should come from the latest profile snapshot:
#   1) METADATA_DATA_PROFILED_FREQUENCY -> top values
#   2) METADATA_DATA_PROFILED -> min/max fallback
PROFILE_CONTEXT = {
    "order_id": {
        "kind": "frequency",
        "values": [("ORD-1001", 3), ("ORD-1002", 2), ("ORD-1003", 2)],
    },
    "customer_id": {
        "kind": "frequency",
        "values": [("CUST-001", 8), ("CUST-002", 6), ("CUST-003", 5)],
    },
    "product_id": {
        "kind": "frequency",
        "values": [("PROD-101", 14), ("PROD-205", 11), ("PROD-330", 9)],
    },
    "order_datetime": {
        "kind": "range",
        "min": "2026-09-01 08:15:00",
        "max": "2026-09-22 20:10:00",
    },
    "quantity": {
        "kind": "range",
        "min": "1",
        "max": "24",
    },
    "unit_price": {
        "kind": "range",
        "min": "9.90",
        "max": "2499.00",
    },
    "order_net_amount": {
        "kind": "range",
        "min": "9.90",
        "max": "4998.00",
    },
    "historical_net_amount": {
        "kind": "range",
        "min": "8.90",
        "max": "4698.00",
    },
    "status": {
        "kind": "frequency",
        "values": [("Completed", 62), ("Pending", 38), ("Cancelled", 20)],
    },
}


RELATIONSHIP_RULES = {
    "composite_unique": {
        "title": "Composite uniqueness",
        "description": "Requires the selected column combination to be unique.",
    },
    "compare_columns": {
        "title": "Compare columns",
        "description": "Compares two columns using an explicit operator.",
    },
    "required_when": {
        "title": "Required when",
        "description": "Requires one or more columns when a condition is met.",
    },
    "conditional_value": {
        "title": "Conditional value",
        "description": "Requires a target value when a condition is met.",
    },
    "referential_columns": {
        "title": "Referential columns",
        "description": "Defines a relationship between this table and another table.",
    },
}


In [ ]:
def esc(value):
    return html.escape(str(value if value is not None else ""))


def find_column(name):
    return next(c for c in state["columns"] if c["name"] == name)


def column_rule_count(column):
    return sum(bool(config.get("enabled", True)) for config in column.get("dq", {}).values())


def total_column_rule_count():
    return sum(column_rule_count(c) for c in state["columns"])


def total_relationship_count():
    return sum(len(items) for items in state["relationships"].values())


def configured_column_summary(column):
    bits = []
    if column["required"]:
        bits.append("Required")
    if column["sensitive_enabled"]:
        bits.append("Sensitive")
    if column_rule_count(column):
        bits.append(f"{column_rule_count(column)} DQ")
    return " · ".join(bits) if bits else "No guardrails"


def section_header(title, suggest_button=None):
    title_html = widgets.HTML(
        f"<div class='fops-field-title'>{esc(title)}</div>",
        layout=widgets.Layout(flex="1 1 auto", min_width="0"),
    )
    children = [title_html]
    if suggest_button is not None:
        children.append(suggest_button)
    box = widgets.HBox(
        children,
        layout=widgets.Layout(
            width="100%",
            align_items="center",
            justify_content="space-between",
            margin="0 0 9px 0",
            padding="7px 9px",
            flex="0 0 auto",
        ),
    )
    box.add_class("fops-title-bar")
    return box


def field_block(title, control, suggest_button=None, note=None):
    children = [section_header(title, suggest_button), control]
    if note:
        children.append(
            widgets.HTML(f"<div class='fops-note' style='margin-top:5px'>{esc(note)}</div>")
        )
    return widgets.VBox(
        children,
        layout=widgets.Layout(
            width="100%",
            margin="0 0 22px 0",
            flex="0 0 auto",
        ),
    )


def suggest_button(description="✨ Suggest"):
    return widgets.Button(
        description=description,
        layout=widgets.Layout(width="105px", height="30px"),
    )


In [ ]:
css = widgets.HTML("""
<style>
.fops-root {
    font-family: "Segoe UI", Arial, sans-serif;
}

.fops-header {
    border: 1px solid #e1e6eb;
    border-radius: 7px;
    padding: 12px 16px;
    background: #fff;
}

.fops-header-row {
    display:flex;
    justify-content:space-between;
    align-items:center;
}

.fops-title {
    color:#0f6cbd;
    font-size:21px;
    font-weight:650;
}

.fops-meta {
    margin-top:4px;
    color:#667085;
    font-size:12px;
}

.fops-contract-badge {
    padding:5px 10px;
    border-radius:14px;
    background:#fff4ce;
    color:#705500;
    font-size:11px;
    font-weight:650;
}

.fops-section-label {
    color:#667085;
    font-size:10px;
    font-weight:700;
    text-transform:uppercase;
    letter-spacing:.055em;
    margin:3px 0 6px 0;
}

.fops-focus-title {
    color:#0f6cbd;
    font-size:23px;
    font-weight:650;
    line-height:1.2;
}

.fops-focus-type {
    color:#667085;
    font-size:12px;
    margin-top:3px;
}

.fops-field-title {
    color:#25354a;
    font-size:13px;
    font-weight:650;
}

.fops-divider {
    border-top:1px solid #edf0f2;
    margin:12px 0;
}

.fops-note {
    color:#667085;
    font-size:11px;
    line-height:1.45;
}

.fops-card {
    border:1px solid #e5e9ed;
    border-radius:6px;
    padding:12px;
    background:#fff;
}

.fops-summary-card {
    border:1px solid #e5e9ed;
    border-radius:6px;
    padding:10px 12px;
    margin-bottom:8px;
    background:#fff;
}

.fops-summary-number {
    color:#0f6cbd;
    font-size:20px;
    font-weight:650;
}

.fops-summary-label {
    color:#667085;
    font-size:11px;
    margin-top:2px;
}

.fops-review-table {
    width:100%;
    border-collapse:collapse;
    font-size:12px;
}

.fops-review-table th {
    color:#667085;
    font-weight:650;
    text-align:left;
    border-bottom:1px solid #e5e9ed;
    padding:7px 8px;
}

.fops-review-table td {
    border-bottom:1px solid #f0f2f4;
    padding:8px;
    vertical-align:top;
}

.fops-detail-label {
    color:#667085;
    font-size:11px;
    margin-bottom:2px;
}

.fops-detail-value {
    color:#25354a;
    font-size:13px;
    margin-bottom:12px;
}

.widget-toggle-buttons .widget-toggle-button {
    min-width: 145px;
}

.widget-label {
    min-width: 0 !important;
}
</style>
""")


In [ ]:
css = widgets.HTML("""
<style>
.fops-root {
    font-family: "Segoe UI", Arial, sans-serif;
    color:#1f2937;
    line-height:1.45;
}

.fops-header {
    border: 1px solid #dfe5eb;
    border-radius: 8px;
    padding: 14px 18px;
    background: #fff;
}

.fops-header-row {
    display:flex;
    justify-content:space-between;
    align-items:flex-start;
    gap:18px;
}

.fops-title {
    color:#0f6cbd;
    font-size:23px;
    font-weight:700;
    line-height:1.2;
}

.fops-meta {
    margin-top:5px;
    color:#667085;
    font-size:12px;
}

.fops-contract-badge {
    padding:6px 11px;
    border-radius:14px;
    background:#fff4ce;
    color:#705500;
    font-size:11px;
    font-weight:700;
    white-space:nowrap;
}

.fops-section-label {
    color:#0f6cbd;
    font-size:11px;
    font-weight:800;
    text-transform:uppercase;
    letter-spacing:.07em;
    margin:0 0 6px 0;
}

.fops-focus-title {
    color:#172b4d;
    font-size:22px;
    font-weight:750;
    line-height:1.2;
    margin:0;
}

.fops-focus-type {
    color:#667085;
    font-size:12px;
    margin-top:5px;
    line-height:1.45;
}

.fops-field-title {
    color:#253858;
    font-size:14px;
    font-weight:700;
    line-height:1.25;
}

.fops-subheading {
    color:#42526e;
    font-size:12px;
    font-weight:700;
    margin:8px 0 6px 0;
}

.fops-divider {
    border-top:1px solid #e6eaef;
    margin:16px 0 18px 0;
}

.fops-note {
    color:#667085;
    font-size:12px;
    line-height:1.5;
}

.fops-card {
    border:1px solid #e1e6eb;
    border-radius:7px;
    padding:14px;
    background:#fff;
    margin-bottom:12px;
}

.fops-rule-card {
    border:1px solid #dfe5eb;
    border-radius:7px;
    padding:12px 14px;
    background:#fbfcfd;
    margin:0 0 10px 0;
}

.fops-rule-title {
    color:#172b4d;
    font-size:13px;
    font-weight:700;
}

.fops-rule-meta {
    color:#667085;
    font-size:11px;
    margin-top:3px;
}

.fops-summary-card {
    border:1px solid #e1e6eb;
    border-radius:7px;
    padding:11px 13px;
    margin-bottom:9px;
    background:#fff;
}

.fops-summary-number {
    color:#0f6cbd;
    font-size:21px;
    font-weight:750;
    line-height:1.1;
}

.fops-summary-label {
    color:#667085;
    font-size:11px;
    margin-top:4px;
}

.fops-review-table {
    width:100%;
    border-collapse:collapse;
    font-size:12px;
}

.fops-review-table th {
    color:#42526e;
    font-weight:700;
    text-align:left;
    border-bottom:2px solid #e5e9ed;
    padding:8px;
}

.fops-review-table td {
    color:#253858;
    border-bottom:1px solid #eef1f4;
    padding:9px 8px;
    vertical-align:top;
}

.fops-detail-label {
    color:#667085;
    font-size:11px;
    font-weight:600;
    text-transform:uppercase;
    letter-spacing:.04em;
    margin-bottom:3px;
}

.fops-detail-value {
    color:#172b4d;
    font-size:13px;
    font-weight:500;
    margin-bottom:15px;
}

.fops-success {
    color:#107c10;
    font-weight:650;
}

.widget-toggle-buttons .widget-toggle-button {
    min-width:145px;
    font-weight:650;
}

.widget-label {
    min-width:0 !important;
}

/* Prevent ipywidgets from shrinking stacked controls into each other. */
.fops-root .widget-box,
.fops-root .jupyter-widgets {
    box-sizing:border-box;
}

.fops-root .widget-vbox > .widget-child,
.fops-root .widget-hbox > .widget-child {
    flex-shrink:0;
}

.fops-selector-shell {
    border:1px solid #dfe5eb;
    border-radius:8px;
    padding:16px 18px;
    background:#f9fbfd;
}
.fops-selector-title {
    color:#172b4d;
    font-size:18px;
    font-weight:750;
    margin-bottom:3px;
}
.fops-selector-note {
    color:#667085;
    font-size:12px;
    margin-bottom:12px;
}
.fops-editing-chip {
    display:inline-block;
    padding:4px 8px;
    border-radius:12px;
    background:#e8f3fb;
    color:#0f6cbd;
    font-size:11px;
    font-weight:700;
    margin-bottom:5px;
}


.fops-title-bar {
    background:#e8f3fb;
    border-left:4px solid #0f6cbd;
    border-radius:4px;
}

.fops-title-bar .fops-field-title {
    color:#0b4f7a;
    font-size:14px;
    font-weight:750;
}

.fops-major-bar {
    background:#0f6cbd;
    color:#ffffff;
    border-radius:5px;
    padding:10px 12px;
    margin:0 0 14px 0;
}

.fops-major-bar .fops-section-label {
    color:#dff1ff;
    margin:0 0 3px 0;
}

.fops-major-bar .fops-focus-title {
    color:#ffffff;
    font-size:20px;
}

.fops-major-bar .fops-focus-type {
    color:#eaf6ff;
    margin-top:4px;
}

.fops-context-bar {
    background:#e7f5ef;
    border-left:4px solid #107c41;
    border-radius:4px;
    padding:9px 10px;
    margin:0 0 12px 0;
}

.fops-context-bar .fops-section-label {
    color:#0b5d35;
    margin:0 0 3px 0;
}

.fops-context-bar .fops-focus-title {
    color:#16392b;
}

.fops-context-bar .fops-focus-type {
    color:#46675a;
}


.fops-root .widget-gridbox {
    overflow: visible !important;
}
.fops-root .widget-button {
    overflow: hidden;
}


.fops-left-label {
    color:#667085;
    font-size:10px;
    font-weight:800;
    text-transform:uppercase;
    letter-spacing:.08em;
    margin:0 0 4px 0;
}

.fops-left-value {
    color:#172b4d;
    font-size:13px;
    font-weight:600;
    line-height:1.4;
}

.fops-left-muted {
    color:#667085;
    font-size:12px;
    line-height:1.45;
}




.fops-readonly-control {
    box-sizing:border-box;
    height:34px;
    min-height:34px;
    display:flex;
    align-items:center;
    padding:0 10px;
    border:1px solid #c8c8c8;
    background:#f7f7f7;
    white-space:nowrap;
}

.fops-root .widget-dropdown select,
.fops-root .widget-combobox input,
.fops-root .widget-text input {
    min-height:34px !important;
    height:34px !important;
    box-sizing:border-box;
}

.fops-root .widget-button {
    min-height:34px !important;
    height:34px !important;
}


.fops-root .widget-checkbox {
    overflow: visible !important;
    min-width: 0 !important;
}

.fops-root .widget-checkbox label {
    white-space: nowrap !important;
    overflow: visible !important;
}

.fops-root .widget-dropdown,
.fops-root .widget-combobox,
.fops-root .widget-text,
.fops-root .widget-textarea {
    overflow: visible !important;
}

</style>
""")


In [ ]:
header = widgets.HTML()

# Environment and logical Fabric stores are inherited from 00_env_config.
# Downstream FabricOps notebooks receive these through FABRIC_CONTEXT.
import builtins

_ACTIVE_CONTEXT = getattr(builtins, "FABRIC_CONTEXT", None) or globals().get("FABRIC_CONTEXT") or {}
ACTIVE_CONFIG = _ACTIVE_CONTEXT.get("config") or globals().get("CONFIG")
ACTIVE_ENV = _ACTIVE_CONTEXT.get("env") or globals().get("ENV") or state["table"]["environment"]
ACTIVE_ENV = str(ACTIVE_ENV)

environment_display = widgets.HTML(
    f"""
    <div class="fops-readonly-control">
        <b>{esc(ACTIVE_ENV)}</b>
        <span class="fops-note" style="margin-left:6px">from 00 Environment</span>
    </div>
    """
)

store_selector = widgets.Dropdown(
    options=[],
    layout=widgets.Layout(width="100%"),
)

schema_selector = widgets.Dropdown(
    options=[],
    layout=widgets.Layout(width="100%"),
)

table_selector = widgets.Combobox(
    options=[],
    placeholder="Search table...",
    ensure_option=True,
    continuous_update=False,
    layout=widgets.Layout(width="100%"),
)

contract_selector = widgets.Dropdown(
    options=[("New draft", "new")],
    value="new",
    layout=widgets.Layout(width="100%"),
)

load_table_button = widgets.Button(
    description="Load Table",
    button_style="primary",
    icon="database",
    layout=widgets.Layout(width="120px", height="34px", flex="0 0 auto"),
)

change_table_button = widgets.Button(
    description="Change table",
    layout=widgets.Layout(width="120px", height="30px"),
)

selector_status = widgets.HTML()

selector_panel = widgets.VBox(
    [
        widgets.HTML("""
            <div class='fops-selector-shell'>
                <div class='fops-section-label'>Data contract</div>
                <div class='fops-selector-title'>Select governed table</div>
                <div class='fops-selector-note'>
                    Select a governed table and Data Contract version.
                </div>
            </div>
        """),
        widgets.GridBox(
            [
                widgets.VBox([
                    widgets.HTML("<div class='fops-field-title'>Environment</div>"),
                    environment_display,
                ], layout=widgets.Layout(gap="5px")),
                widgets.VBox([
                    widgets.HTML("<div class='fops-field-title'>Fabric store</div>"),
                    store_selector,
                ], layout=widgets.Layout(gap="5px")),
                widgets.VBox([
                    widgets.HTML("<div class='fops-field-title'>Schema</div>"),
                    schema_selector,
                ], layout=widgets.Layout(gap="5px")),
                widgets.VBox([
                    widgets.HTML("<div class='fops-field-title'>Table</div>"),
                    table_selector,
                ], layout=widgets.Layout(gap="5px")),
                widgets.VBox([
                    widgets.HTML("<div class='fops-field-title'>Contract</div>"),
                    contract_selector,
                ], layout=widgets.Layout(gap="5px")),
                widgets.VBox([
                    widgets.HTML("<div class='fops-field-title'>&nbsp;</div>"),
                    load_table_button,
                ], layout=widgets.Layout(
                    width="120px",
                    overflow="hidden",
                    align_items="flex-start",
                    gap="5px",
                )),
            ],
            layout=widgets.Layout(
                width="100%",
                grid_template_columns="170px minmax(205px,1fr) minmax(145px,.7fr) minmax(220px,1.05fr) minmax(175px,.8fr) 120px",
                grid_gap="12px",
                align_items="flex-end",
                overflow="visible",
            ),
        ),
        selector_status,
    ],
    layout=widgets.Layout(width="100%", gap="9px"),
)

top_nav = widgets.ToggleButtons(
    options=[
        ("Table", "table"),
        ("Columns", "columns"),
        ("Advanced", "relationships"),
        ("Manifest & Freeze", "review"),
    ],
    value="table",
    layout=widgets.Layout(width="620px"),
)

left = widgets.VBox(
    layout=widgets.Layout(
        width="100%",
        border="1px solid #e1e6eb",
        padding="16px",
        align_items="stretch",
    )
)

right = widgets.VBox(
    layout=widgets.Layout(
        width="100%",
        border="1px solid #e1e6eb",
        padding="20px 24px",
        align_items="stretch",
    )
)

workspace = widgets.GridBox(
    [left, right],
    layout=widgets.Layout(
        width="100%",
        grid_template_columns="minmax(250px, 27fr) minmax(0, 73fr)",
        grid_gap="12px",
        align_items="flex-start",
    ),
)

status = widgets.HTML(
    layout=widgets.Layout(width="100%", min_height="24px")
)

editor_shell = widgets.VBox(
    [
        widgets.HBox(
            [top_nav],
            layout=widgets.Layout(
                width="100%",
                justify_content="center",
                margin="4px 0 6px 0",
            ),
        ),
        workspace,
        status,
    ],
    layout=widgets.Layout(width="100%", gap="7px"),
)

selection = {
    "column": state["columns"][0]["name"],
    "relationship": "composite_unique",
    "relationship_index": None,
    "review": "table",
}


In [ ]:
def render_header():
    table = state["table"]
    contract = state["contract"]

    described = sum(bool(c["description"]) for c in state["columns"])
    required = sum(bool(c["required"]) for c in state["columns"])
    sensitive = sum(bool(c["sensitive_enabled"]) for c in state["columns"])

    header.value = f"""
    <div class="fops-header">
        <div class="fops-header-row">
            <div>
                <div class="fops-title">{esc(table["table_name"])}</div>
                <div class="fops-meta">
                    {esc(table["store_type"])} · {esc(table["schema"])}.{esc(table["table_name"])}
                </div>
                <div class="fops-meta" style="margin-top:5px;">
                    {table["rows"]:,} rows
                    &nbsp; · &nbsp; {table["columns"]} columns
                    &nbsp; · &nbsp; {esc(table["load_strategy"])}
                    &nbsp; · &nbsp; profiled {esc(table["profiled_at"])}
                </div>
                <div class="fops-meta" style="margin-top:5px;">
                    <b>{described}</b> described
                    &nbsp; · &nbsp; <b>{required}</b> required
                    &nbsp; · &nbsp; <b>{sensitive}</b> sensitive
                    &nbsp; · &nbsp; <b>{total_column_rule_count()}</b> column DQ
                    &nbsp; · &nbsp; <b>{total_relationship_count()}</b> relationships
                </div>
            </div>
            <div class="fops-contract-badge">
                {esc(contract["status"])} · v{contract["version"]}
            </div>
        </div>
    </div>
    """


In [ ]:
def catalogue_matches(**filters):
    rows = TABLE_CATALOGUE
    for key, value in filters.items():
        if value not in (None, ""):
            rows = [r for r in rows if r[key] == value]
    return rows


def configured_fabric_stores():
    """Return logical stores from the active 00_env_config Fabric configuration."""
    if ACTIVE_CONFIG is not None:
        try:
            return dict(ACTIVE_CONFIG.path_config.paths[ACTIVE_ENV])
        except (AttributeError, KeyError, TypeError):
            pass

    # Self-contained prototype fallback only.
    fallback = {}
    for row in TABLE_CATALOGUE:
        if row["environment"] != ACTIVE_ENV:
            continue
        fallback.setdefault(
            row["store"],
            type(
                "PrototypeFabricStore",
                (),
                {
                    "kind": row["store_type"].lower(),
                    "schema": row["schema"],
                    "schema_enabled": True,
                },
            )(),
        )
    return fallback


def selected_store_config():
    return configured_fabric_stores().get(store_selector.value)


def selected_store_kind():
    store = selected_store_config()
    kind = getattr(store, "kind", "") if store is not None else ""
    return str(kind).strip().lower()


CONTRACT_CATALOGUE = {
    "curated_orders": [
        {"version": 4, "status": "Frozen", "is_active": False},
        {"version": 3, "status": "Active", "is_active": True},
        {"version": 2, "status": "Superseded", "is_active": False},
        {"version": 1, "status": "Superseded", "is_active": False},
    ],
    "orders": [
        {"version": 2, "status": "Active", "is_active": True},
        {"version": 1, "status": "Superseded", "is_active": False},
    ],
    "customer_summary": [
        {"version": 1, "status": "Frozen", "is_active": False},
    ],
}


def refresh_contract_selector():
    table_name = str(table_selector.value or "").strip()
    versions = CONTRACT_CATALOGUE.get(table_name, [])

    options = [("New draft", "new")]
    options.extend(
        (f"v{row['version']} · {row['status']}", f"v{row['version']}")
        for row in sorted(versions, key=lambda r: int(r["version"]), reverse=True)
    )

    preferred = contract_selector.value or "new"
    values = [value for _, value in options]

    contract_selector.index = None
    contract_selector.options = options
    contract_selector.value = preferred if preferred in values else "new"


_selector_refreshing = False


def _set_dropdown_options(widget, options, preferred=None):
    """Safely replace Dropdown options without retaining a stale selection."""
    options = list(options)
    values = [item[1] if isinstance(item, tuple) else item for item in options]

    # Important: clear by index BEFORE replacing options.
    # Setting .options first can immediately validate the old value and raise
    # "Invalid selection: value not found".
    widget.index = None
    widget.options = options

    if preferred in values:
        widget.value = preferred
    elif values:
        widget.index = 0


def _set_combobox_options(widget, options, preferred=""):
    """Safely replace Combobox options without retaining a stale selection."""
    options = list(options)

    # Allow clearing before replacing the option list, then restore strict mode.
    original_ensure = table_selector.ensure_option
    table_selector.ensure_option = False
    table_selector.value = ""
    table_selector.options = options

    if preferred in options:
        table_selector.value = preferred

    table_selector.ensure_option = original_ensure


def refresh_table_selector_options(*_, preserve_table=True):
    """Refresh dependent selectors without recursive or invalid selections."""
    global _selector_refreshing
    if _selector_refreshing:
        return

    _selector_refreshing = True
    try:
        stores = configured_fabric_stores()

        store_options = []
        for name, store in stores.items():
            kind = str(getattr(store, "kind", "") or "").strip()
            label = f"{name} · {kind.title()}" if kind else name
            store_options.append((label, name))

        preferred_store = store_selector.value or state["table"].get("store")
        _set_dropdown_options(
            store_selector,
            store_options,
            preferred=preferred_store,
        )

        if store_selector.value is None:
            _set_dropdown_options(schema_selector, [], preferred=None)
            _set_combobox_options(table_selector, [], preferred="")
            return

        store_kind = selected_store_kind()

        matches = catalogue_matches(
            environment=ACTIVE_ENV,
            store=store_selector.value,
        )
        if store_kind:
            matches = [
                row
                for row in matches
                if str(row["store_type"]).lower() == store_kind
            ]

        configured_store = selected_store_config()
        configured_schema = (
            getattr(configured_store, "schema", None)
            if configured_store is not None
            else None
        )

        schemas = sorted({row["schema"] for row in matches})

        # A configured store schema is only selectable here when the prototype
        # catalogue actually contains it. This avoids a Dropdown TraitError.
        preferred_schema = schema_selector.value or state["table"].get("schema")
        if preferred_schema not in schemas:
            preferred_schema = configured_schema if configured_schema in schemas else None

        _set_dropdown_options(
            schema_selector,
            schemas,
            preferred=preferred_schema,
        )

        if schema_selector.value is None:
            _set_combobox_options(table_selector, [], preferred="")
            return

        tables = sorted({
            row["table_name"]
            for row in matches
            if row["schema"] == schema_selector.value
        })

        preferred_table = table_selector.value if preserve_table else ""
        if not preferred_table:
            preferred_table = state["table"].get("table_name", "")

        _set_combobox_options(
            table_selector,
            tables,
            preferred=preferred_table,
        )
        refresh_contract_selector()

    finally:
        _selector_refreshing = False


def current_catalogue_record():
    store_kind = selected_store_kind()
    matches = catalogue_matches(
        environment=ACTIVE_ENV,
        store=store_selector.value,
        schema=schema_selector.value,
        table_name=table_selector.value,
    )
    if store_kind:
        matches = [
            r for r in matches
            if str(r["store_type"]).lower() == store_kind
        ]
    return matches[0] if matches else None


def show_table_selector():
    selector_panel.layout.display = ""
    editor_shell.layout.display = "none"


def show_editor():
    selector_panel.layout.display = "none"
    editor_shell.layout.display = ""


def load_selected_table(_=None):
    record = current_catalogue_record()
    if not record:
        selector_status.value = "<span style='color:#b42318'>No matching governed table found.</span>"
        return

    state["table"].update(record)

    selected_contract = contract_selector.value or "new"
    versions = CONTRACT_CATALOGUE.get(record["table_name"], [])

    if selected_contract == "new":
        state["contract"]["status"] = "DRAFT"
        state["contract"]["version"] = max(
            [int(row["version"]) for row in versions],
            default=0,
        ) + 1
    else:
        version = int(str(selected_contract).lstrip("v"))
        selected_version = next(
            (row for row in versions if int(row["version"]) == version),
            {"version": version, "status": "Frozen", "is_active": False},
        )
        state["contract"]["status"] = str(selected_version["status"]).upper()
        state["contract"]["version"] = version

    # Prototype only:
    # the selected table drives the physical context shown by the UI.
    # In production, table_id would hydrate the real schema/profile/contract
    # from FabricOps metadata instead of using the in-notebook mock contract.
    if record["table_name"] == "curated_orders":
        state["table"]["columns"] = len(state["columns"])

    selection["column"] = state["columns"][0]["name"]
    selection["relationship"] = "composite_unique"
    selection["relationship_index"] = None
    selection["review"] = "table"

    render_header()
    render_current_view()
    show_editor()
    selector_status.value = ""


def on_store_change(change):
    if change["name"] == "value" and not _selector_refreshing:
        refresh_table_selector_options(preserve_table=False)


def on_schema_change(change):
    if change["name"] == "value" and not _selector_refreshing:
        refresh_table_selector_options(preserve_table=False)


load_table_button.on_click(load_selected_table)
change_table_button.on_click(lambda _: show_table_selector())

# Initialise once before attaching observers.
refresh_table_selector_options(preserve_table=False)

store_selector.observe(on_store_change, names="value")
schema_selector.observe(on_schema_change, names="value")
table_selector.observe(
    lambda change: refresh_contract_selector() if change["name"] == "value" else None,
    names="value",
)


In [ ]:
table_description = widgets.Textarea(
    value=state["table_definition"]["description"],
    layout=widgets.Layout(width="100%", height="78px"),
)

table_description_ai = suggest_button()

table_classification = widgets.Dropdown(
    value=state["table_definition"]["classification"],
    options=[
        ("Not classified", ""),
        ("Public", "Public"),
        ("Internal", "Internal"),
        ("Confidential", "Confidential"),
        ("Restricted", "Restricted"),
    ],
    layout=widgets.Layout(width="260px"),
)


freshness_enabled = widgets.Checkbox(
    value=state["table_definition"]["freshness_enabled"],
    description="Enabled",
    indent=False,
    layout=widgets.Layout(width="100px"),
)

freshness_column = widgets.Dropdown(
    value=state["table_definition"]["freshness_column"],
    options=[c["name"] for c in state["columns"]],
    layout=widgets.Layout(width="250px"),
)

freshness_age = widgets.FloatText(
    value=state["table_definition"]["freshness_age"],
    layout=widgets.Layout(width="90px"),
)

freshness_unit = widgets.Dropdown(
    value=state["table_definition"]["freshness_unit"],
    options=["Minutes", "Hours", "Days"],
    layout=widgets.Layout(width="110px"),
)

freshness_action = widgets.Checkbox(
    value=state["table_definition"]["freshness_block"],
    description="Block on failure",
    indent=False,
    layout=widgets.Layout(width="150px"),
)

drift_enabled = widgets.Checkbox(
    value=state["table_definition"]["source_drift_enabled"],
    description="Enabled",
    indent=False,
    layout=widgets.Layout(width="100px"),
)

drift_partition = widgets.Dropdown(
    value=state["table_definition"]["source_drift_partition"],
    options=[("No partition", "")] + [(c["name"], c["name"]) for c in state["columns"]],
    layout=widgets.Layout(width="240px"),
)

drift_change = widgets.Dropdown(
    value=state["table_definition"]["source_drift_change"],
    options=[("No change column", "")] + [(c["name"], c["name"]) for c in state["columns"]],
    layout=widgets.Layout(width="240px"),
)

drift_action = widgets.Checkbox(
    value=state["table_definition"]["source_drift_block"],
    description="Block on failure",
    indent=False,
    layout=widgets.Layout(width="150px"),
)

save_table = widgets.Button(
    description="Save Table",
    button_style="primary",
    layout=widgets.Layout(width="130px", height="34px"),
)


def render_table():
    table = state["table"]
    contract = state["contract"]

    classification = state["table_definition"].get("classification") or "Not classified"
    refresh_frequency = state["table_definition"].get("refresh_frequency") or "Not defined"

    schema_on = any(c.get("required") for c in state["columns"])
    freshness_on = bool(state["table_definition"].get("freshness_enabled"))
    sensitive_on = any(c.get("sensitive_enabled") for c in state["columns"])
    source_drift_on = bool(state["table_definition"].get("source_drift_enabled"))
    dq_on = total_column_rule_count() > 0 or total_relationship_count() > 0

    guardrail_rows = [
        ("Schema", schema_on),
        ("Freshness", freshness_on),
        ("Sensitive Data", sensitive_on),
        ("Source Drift", source_drift_on),
        ("Data Quality", dq_on),
    ]

    guardrail_status_html = "".join(
        f"""
        <div style='display:flex;justify-content:space-between;gap:12px;padding:3px 0'>
            <span class='fops-left-muted'>{esc(name)}</span>
            <span class='fops-left-value' style='font-size:12px'>
                {"Enabled" if enabled else "Disabled"}
            </span>
        </div>
        """
        for name, enabled in guardrail_rows
    )

    left.children = [
        widgets.HTML(
            f"""
            <div class='fops-context-bar'>
                <div class='fops-focus-title' style='font-size:18px'>
                    {esc(table["schema"])}.{esc(table["table_name"])}
                </div>
                <div class='fops-left-muted' style='margin-top:6px'>
                    {esc(table["environment"])} · {esc(table["store"])} · {esc(table["store_type"])}
                </div>
                <div class='fops-left-muted' style='margin-top:4px'>
                    {table["rows"]:,} rows · {table["columns"]} columns
                </div>
                <div class='fops-left-muted' style='margin-top:4px'>
                    profiled {esc(table["profiled_at"])}
                </div>
            </div>

            <div class='fops-divider'></div>

            <div class='fops-left-label'><b>Contract</b></div>
            <div class='fops-left-value'>
                v{contract["version"]} · {esc(contract["status"])}
            </div>

            <div style='margin-top:12px'>
                <div class='fops-left-label'><b>Loading Strategy</b></div>
                <div class='fops-left-value'>{esc(table["load_strategy"])}</div>
            </div>

            <div style='margin-top:12px'>
                <div class='fops-left-label'><b>Refresh Frequency</b></div>
                <div class='fops-left-value'>{esc(refresh_frequency)}</div>
            </div>

            <div style='margin-top:12px'>
                <div class='fops-left-label'><b>Classification</b></div>
                <div class='fops-left-value'>{esc(classification)}</div>
            </div>

            <div class='fops-divider'></div>

            <div class='fops-left-label'><b>Guardrails</b></div>
            <div style='margin-top:6px'>
                {guardrail_status_html}
            </div>
            """
        ),
        widgets.HTML("<div style='height:8px'></div>"),
        change_table_button,
    ]

    freshness_body = widgets.VBox([
        widgets.HTML("<div class='fops-field-title'>Freshness Check</div>"),
        freshness_enabled,
        freshness_action,
        widgets.VBox([
            widgets.HTML("<div class='fops-note'>Column</div>"),
            freshness_column,
        ], layout=widgets.Layout(width="320px", max_width="100%", gap="4px")),
        widgets.VBox([
            widgets.HTML("<div class='fops-note'>Maximum age</div>"),
            freshness_age,
        ], layout=widgets.Layout(width="140px", max_width="100%", gap="4px")),
        widgets.VBox([
            widgets.HTML("<div class='fops-note'>Unit</div>"),
            freshness_unit,
        ], layout=widgets.Layout(width="180px", max_width="100%", gap="4px")),
    ], layout=widgets.Layout(width="100%", gap="8px"))

    drift_body = widgets.VBox([
        widgets.HTML("<div class='fops-field-title'>Source Drift Check</div>"),
        drift_enabled,
        drift_action,
        widgets.VBox([
            widgets.HTML("<div class='fops-note'>Partition</div>"),
            drift_partition,
        ], layout=widgets.Layout(width="320px", max_width="100%", gap="4px")),
        widgets.VBox([
            widgets.HTML("<div class='fops-note'>Change column</div>"),
            drift_change,
        ], layout=widgets.Layout(width="320px", max_width="100%", gap="4px")),
    ], layout=widgets.Layout(width="100%", gap="8px"))

    right.children = [
        widgets.VBox(
            [
                widgets.HTML("<div class='fops-field-title'>Classification</div>"),
                table_classification,
            ],
            layout=widgets.Layout(
                width="320px",
                max_width="100%",
                gap="8px",
                margin="0 0 16px 0",
            ),
        ),
        widgets.VBox(
            [
                section_header("Description", table_description_ai),
                table_description,
            ],
            layout=widgets.Layout(
                width="100%",
                gap="0px",
                margin="0 0 22px 0",
            ),
        ),
        widgets.VBox(
            [freshness_body],
            layout=widgets.Layout(
                width="100%",
                margin="0 0 22px 0",
                padding="10px 12px",
                border="1px solid #dfe5eb",
            ),
        ),
        widgets.VBox(
            [drift_body],
            layout=widgets.Layout(
                width="100%",
                margin="0 0 22px 0",
                padding="10px 12px",
                border="1px solid #dfe5eb",
            ),
        ),
        widgets.HBox([save_table], layout=widgets.Layout(justify_content="flex-end")),
    ]


def save_table_clicked(_):
    td = state["table_definition"]
    td["description"] = table_description.value
    td["classification"] = table_classification.value
    td["freshness_enabled"] = freshness_enabled.value
    td["freshness_column"] = freshness_column.value
    td["freshness_age"] = freshness_age.value
    td["freshness_unit"] = freshness_unit.value
    td["freshness_block"] = freshness_action.value
    td["source_drift_enabled"] = drift_enabled.value
    td["source_drift_partition"] = drift_partition.value
    td["source_drift_change"] = drift_change.value
    td["source_drift_block"] = drift_action.value
    render_header()
    status.value = "<span class='fops-success'>✓ Table contract saved</span>"

save_table.on_click(save_table_clicked)


In [ ]:
column_search = widgets.Text(
    placeholder="Search columns",
    layout=widgets.Layout(width="100%"),
)

column_selector = widgets.Select(
    options=[],
    rows=14,
    layout=widgets.Layout(width="100%", height="405px"),
)

column_description = widgets.Textarea(
    layout=widgets.Layout(width="100%", height="58px")
)
column_description_ai = suggest_button()

column_classification = widgets.Dropdown(
    options=[
        ("Not classified", ""),
        ("Public", "Public"),
        ("Internal", "Internal"),
        ("Confidential", "Confidential"),
        ("Restricted", "Restricted"),
    ],
    layout=widgets.Layout(width="100%"),
)

column_required = widgets.Checkbox(
    value=False,
    description="Required",
    indent=False,
    layout=widgets.Layout(width="110px"),
)

sensitive_enabled = widgets.Checkbox(
    value=False,
    description="Enabled",
    indent=False,
    layout=widgets.Layout(width="100px"),
)

pii_assessment = widgets.HTML(
    "<div class='fops-left-muted'>AI assessment: <b>Not assessed</b></div>"
)
pii_recommendation = widgets.HTML(
    "<div class='fops-left-muted'>AI recommends: <b>None yet</b></div>"
)

treatment = widgets.Dropdown(
    options=["tokenize", "mask", "bucket", "remove"],
    layout=widgets.Layout(width="220px"),
)

sensitive_action = widgets.Checkbox(
    value=True,
    description="Block on failure",
    indent=False,
    layout=widgets.Layout(width="150px"),
)


column_rule_selector = widgets.Select(
    rows=7,
    layout=widgets.Layout(width="100%", height="205px"),
)

column_rule_editor = widgets.VBox()
column_rule_ai = suggest_button("✨ Suggest rules")

save_column = widgets.Button(
    description="Save Column",
    button_style="primary",
    layout=widgets.Layout(width="130px", height="34px"),
)

selected_column_rule = {"key": "missing_values"}


def applicable_column_rules(column):
    result = []
    for key, meta in COLUMN_RULES.items():
        allowed = meta["types"]
        if allowed == "all" or column["type"] in allowed:
            result.append((meta["title"], key))
    return result


def refresh_column_selector():
    query = column_search.value.strip().lower()
    options = []
    for column in state["columns"]:
        if query and query not in column["name"].lower():
            continue
        required = "   *" if column["required"] else ""
        label = f"{column['name']}   ·   {column['type']}{required}"
        options.append((label, column["name"]))

    current = selection["column"]
    column_selector.options = options
    values = [value for _, value in options]
    column_selector.value = current if current in values else (values[0] if values else None)


def render_column_rule_editor():
    if not selection["column"]:
        column_rule_editor.children = ()
        return

    column = find_column(selection["column"])
    key = selected_column_rule["key"]
    meta = COLUMN_RULES[key]
    config = column.get("dq", {}).get(key, {})

    enabled = widgets.Checkbox(
        value=bool(config.get("enabled", False)),
        description="Enabled",
        indent=False,
        layout=widgets.Layout(width="100px"),
    )
    block = widgets.Checkbox(
        value=bool(config.get("block", True)),
        description="Block on failure",
        indent=False,
        layout=widgets.Layout(width="150px"),
    )

    controls = [
        widgets.HBox(
            [
                widgets.HTML(
                    f"<div class='fops-field-title' style='min-width:170px'>{esc(meta['title'])}</div>"
                ),
                enabled,
                block,
            ],
            layout=widgets.Layout(
                width="100%",
                gap="8px",
                align_items="center",
                flex_flow="row wrap",
            ),
        ),
        widgets.HTML(
            f"<div class='fops-note' style='margin:2px 0 8px 0'>{esc(meta['description'])}</div>"
        ),
    ]

    payload = {}

    if key == "missing_values":
        value = widgets.FloatText(
            value=float(config.get("maximum_null_percent", 0.0)),
            layout=widgets.Layout(width="140px"),
        )
        payload["maximum_null_percent"] = value
        controls.append(
            widgets.VBox(
                [
                    widgets.HTML("<div class='fops-note'>Maximum null %</div>"),
                    value,
                ],
                layout=widgets.Layout(width="180px", max_width="100%", gap="4px"),
            )
        )

    elif key in {"accepted_values", "blocked_values"}:
        value = widgets.Text(
            value=config.get("values", ""),
            placeholder="Pending, Completed, Cancelled",
            layout=widgets.Layout(width="100%"),
        )
        payload["values"] = value
        controls.append(
            widgets.VBox(
                [
                    widgets.HTML("<div class='fops-note'>Values</div>"),
                    value,
                ],
                layout=widgets.Layout(width="100%", gap="4px"),
            )
        )

    elif key == "value_range":
        minimum = widgets.Text(
            value=str(config.get("minimum", "")),
            placeholder="No minimum",
            layout=widgets.Layout(width="220px"),
        )
        maximum = widgets.Text(
            value=str(config.get("maximum", "")),
            placeholder="No maximum",
            layout=widgets.Layout(width="220px"),
        )
        min_inc = widgets.Checkbox(
            value=bool(config.get("minimum_inclusive", True)),
            description="Minimum inclusive",
            indent=False,
        )
        max_inc = widgets.Checkbox(
            value=bool(config.get("maximum_inclusive", True)),
            description="Maximum inclusive",
            indent=False,
        )
        payload.update({
            "minimum": minimum,
            "maximum": maximum,
            "minimum_inclusive": min_inc,
            "maximum_inclusive": max_inc,
        })
        controls.extend([
            widgets.VBox(
                [
                    widgets.HTML("<div class='fops-note'>Minimum</div>"),
                    minimum,
                    min_inc,
                ],
                layout=widgets.Layout(width="240px", max_width="100%", gap="4px"),
            ),
            widgets.VBox(
                [
                    widgets.HTML("<div class='fops-note'>Maximum</div>"),
                    maximum,
                    max_inc,
                ],
                layout=widgets.Layout(width="240px", max_width="100%", gap="4px"),
            ),
        ])

    elif key == "text_pattern":
        pattern = widgets.Text(
            value=config.get("pattern", ""),
            placeholder="Regex pattern",
            layout=widgets.Layout(width="100%"),
        )
        payload["pattern"] = pattern
        controls.append(
            widgets.VBox(
                [
                    widgets.HTML("<div class='fops-note'>Pattern</div>"),
                    pattern,
                ],
                layout=widgets.Layout(width="100%", gap="4px"),
            )
        )

    save_rule = widgets.Button(
        description="Save rule",
        button_style="primary",
        layout=widgets.Layout(width="110px", height="32px"),
    )

    remove_rule = widgets.Button(
        description="Remove",
        layout=widgets.Layout(width="95px", height="32px"),
    )

    def save_rule_clicked(_):
        column.setdefault("dq", {})
        new_config = {"enabled": enabled.value, "block": block.value}
        for name, widget in payload.items():
            new_config[name] = widget.value
        if enabled.value:
            column["dq"][key] = new_config
        else:
            column["dq"].pop(key, None)
        refresh_column_selector()
        render_header()
        render_column(selection["column"])
        status.value = f"<span class='fops-success'>✓ {esc(meta['title'])} saved for {esc(column['name'])}</span>"

    def remove_rule_clicked(_):
        column.setdefault("dq", {}).pop(key, None)
        refresh_column_selector()
        render_header()
        render_column(selection["column"])
        status.value = f"<span class='fops-success'>✓ {esc(meta['title'])} removed from {esc(column['name'])}</span>"

    save_rule.on_click(save_rule_clicked)
    remove_rule.on_click(remove_rule_clicked)

    controls.append(
        widgets.HBox([save_rule, remove_rule], layout=widgets.Layout(gap="8px"))
    )

    column_rule_editor.children = tuple(controls)
    column_rule_editor.layout = widgets.Layout(
        width="100%",
        gap="8px",
        padding="8px 0 0 0",
    )



def observed_profile_widget(column_name):
    """Render profile evidence for the selected column."""
    profile = PROFILE_CONTEXT.get(column_name) or {}

    if profile.get("kind") == "frequency":
        values = list(profile.get("values") or [])[:3]
        if values:
            total = sum(count for _, count in values) or 1
            rows = "".join(
                f"""
                <div style='display:flex;justify-content:space-between;gap:16px;padding:3px 0'>
                    <span>{esc(value)}</span>
                    <span class='fops-left-muted'>{count}</span>
                </div>
                """
                for value, count in values
            )
            return widgets.HTML(
                f"""
                <div class='fops-field-title'>Observed values</div>
                <div class='fops-note' style='margin:3px 0 6px 0'>
                    Top values from the latest profiled frequency.
                </div>
                <div style='max-width:420px'>{rows}</div>
                """
            )

    if profile.get("kind") == "range":
        minimum = profile.get("min")
        maximum = profile.get("max")
        if minimum is not None or maximum is not None:
            return widgets.HTML(
                f"""
                <div class='fops-field-title'>Observed range</div>
                <div class='fops-note' style='margin:3px 0 6px 0'>
                    Min / max from the latest profile snapshot.
                </div>
                <div style='display:grid;grid-template-columns:70px minmax(0,1fr);gap:4px 12px;max-width:420px'>
                    <b>Min</b><span>{esc(minimum if minimum is not None else "—")}</span>
                    <b>Max</b><span>{esc(maximum if maximum is not None else "—")}</span>
                </div>
                """
            )

    return widgets.HTML(
        """
        <div class='fops-field-title'>Observed values</div>
        <div class='fops-note' style='margin-top:3px'>No profile values available.</div>
        """
    )


def render_column(name):
    column = find_column(name)
    selection["column"] = name

    column_description.value = column["description"]
    column_classification.value = column["classification"]
    column_required.value = column["required"]
    sensitive_enabled.value = column["sensitive_enabled"]
    treatment.value = column["treatment"] if column["treatment"] else "tokenize"
    sensitive_action.value = column["sensitive_block"]

    rules = applicable_column_rules(column)
    column_rule_selector.options = rules

    rule_keys = [value for _, value in rules]
    if selected_column_rule["key"] not in rule_keys:
        selected_column_rule["key"] = rule_keys[0]
    column_rule_selector.value = selected_column_rule["key"]

    left.children = [
        widgets.HTML("<div class='fops-section-label'>Columns</div>"),
        column_search,
        column_selector,
    ]

    sensitive_body = widgets.VBox([
        widgets.HBox(
            [
                widgets.HTML("<div class='fops-field-title' style='min-width:150px'>Sensitive Data</div>"),
                sensitive_enabled,
                sensitive_action,
            ],
            layout=widgets.Layout(
                width="100%",
                gap="8px",
                align_items="center",
                flex_flow="row wrap",
            ),
        ),
        widgets.HTML(
            "<div class='fops-note' style='margin:8px 0 4px 0'>"
            "Future AI assessment will classify this column as Direct PII, Indirect PII, or Not PII "
            "using configured metadata and prompt guidance, then recommend a treatment for review."
            "</div>"
        ),
        pii_assessment,
        pii_recommendation,
        widgets.HBox(
            [
                widgets.HTML("<div style='width:125px;min-width:125px'>Treatment</div>"),
                treatment,
            ],
            layout=widgets.Layout(
                gap="8px",
                align_items="center",
                margin="6px 0 0 0",
            ),
        ),
    ], layout=widgets.Layout(gap="6px"))

    rule_panel = widgets.VBox([
        section_header("Column data quality", column_rule_ai),
        widgets.HTML(
            "<div class='fops-note' style='margin-bottom:8px'>"
            ""
            "</div>"
        ),
        widgets.GridBox(
            [column_rule_selector, column_rule_editor],
            layout=widgets.Layout(
                width="100%",
                grid_template_columns="minmax(190px, 32fr) minmax(0, 68fr)",
                grid_gap="14px",
            ),
        ),
    ], layout=widgets.Layout(width="100%", margin="0 0 18px 0"))

    right.children = [
        widgets.HBox(
            [
                widgets.HTML("<div class='fops-field-title' style='min-width:150px'>Schema</div>"),
                column_required,
            ],
            layout=widgets.Layout(
                width="100%",
                gap="8px",
                align_items="center",
                margin="0 0 18px 0",
            ),
        ),
        widgets.VBox(
            [
                widgets.HTML("<div class='fops-field-title'>Classification</div>"),
                column_classification,
            ],
            layout=widgets.Layout(
                width="320px",
                max_width="100%",
                gap="8px",
                margin="0 0 16px 0",
            ),
        ),
        widgets.VBox(
            [
                section_header("Description", column_description_ai),
                column_description,
            ],
            layout=widgets.Layout(
                width="100%",
                gap="0px",
                margin="0 0 18px 0",
            ),
        ),
        widgets.VBox(
            [observed_profile_widget(column["name"])],
            layout=widgets.Layout(
                width="100%",
                margin="0 0 18px 0",
                padding="10px 12px",
                border="1px solid #e1e6eb",
            ),
        ),
        widgets.VBox(
            [sensitive_body],
            layout=widgets.Layout(
                width="100%",
                margin="0 0 22px 0",
                padding="10px 12px",
                border="1px solid #dfe5eb",
            ),
        ),
        rule_panel,
        widgets.HBox([save_column], layout=widgets.Layout(justify_content="flex-end")),
    ]

    render_column_rule_editor()


def save_column_clicked(_):
    column = find_column(selection["column"])
    column["description"] = column_description.value
    column["classification"] = column_classification.value
    column["required"] = column_required.value
    column["sensitive_enabled"] = sensitive_enabled.value
    column["treatment"] = treatment.value if sensitive_enabled.value else ""
    column["sensitive_block"] = sensitive_action.value
    refresh_column_selector()
    render_header()
    status.value = f"<span class='fops-success'>✓ {esc(column['name'])} saved</span>"

save_column.on_click(save_column_clicked)


def column_changed(change):
    if change["name"] == "value" and change["new"]:
        render_column(change["new"])

column_selector.observe(column_changed, names="value")


def column_rule_changed(change):
    if change["name"] == "value" and change["new"]:
        selected_column_rule["key"] = change["new"]
        render_column_rule_editor()

column_rule_selector.observe(column_rule_changed, names="value")


column_search.observe(
    lambda change: refresh_column_selector() if change["name"] == "value" else None,
    names="value",
)


In [ ]:
relationship_selector = widgets.Select(
    rows=5,
    layout=widgets.Layout(width="100%", height="220px"),
)

relationship_editor = widgets.VBox()

relationship_ai = suggest_button("✨ Suggest rules")


def refresh_relationship_selector():
    relationship_selector.options = [
        (
            f"{meta['title']}   ·   {len(state['relationships'].get(key, []))} configured",
            key,
        )
        for key, meta in RELATIONSHIP_RULES.items()
    ]
    relationship_selector.value = selection["relationship"]


def render_relationship_editor():
    key = selection["relationship"]
    meta = RELATIONSHIP_RULES[key]
    existing = state["relationships"].get(key, [])

    title = widgets.HTML(
        f"<div class='fops-focus-title'>{esc(meta['title'])}</div>"
        f"<div class='fops-focus-type'>{esc(meta['description'])}</div>"
        "<div class='fops-divider'></div>"
    )

    controls = [title]

    if key == "composite_unique":
        current = existing[0] if existing else {"columns": [], "block": True}
        checks = []
        for c in state["columns"]:
            checks.append(
                widgets.Checkbox(
                    value=c["name"] in current["columns"],
                    description=f"{c['name']} · {c['type']}",
                    indent=False,
                    layout=widgets.Layout(width="100%"),
                )
            )

        block = widgets.Checkbox(
            value=current.get("block", True),
            description="Block",
            indent=False,
        )

        save = widgets.Button(
            description="Save combination",
            button_style="primary",
            layout=widgets.Layout(width="150px", height="34px"),
        )

        def save_clicked(_):
            selected = [
                c["name"]
                for c, checkbox in zip(state["columns"], checks)
                if checkbox.value
            ]
            if len(selected) < 2:
                status.value = "<span style='color:#b42318'>Select at least two columns.</span>"
                return
            state["relationships"][key] = [{
                "columns": selected,
                "block": block.value,
            }]
            refresh_relationship_selector()
            render_header()
            render_relationship_editor()
            status.value = "<span class='fops-success'>✓ Composite uniqueness saved</span>"

        save.on_click(save_clicked)

        controls.extend([
            widgets.HTML("<div class='fops-field-title'>Columns in combination</div>"),
            widgets.VBox(checks),
            block,
            widgets.HBox([relationship_ai, save], layout=widgets.Layout(gap="8px")),
        ])

    elif key == "compare_columns":
        current = existing[0] if existing else {}
        left_col = widgets.Dropdown(
            options=[c["name"] for c in state["columns"]],
            value=current.get("left", state["columns"][0]["name"]),
            layout=widgets.Layout(width="220px"),
        )
        operator = widgets.Dropdown(
            options=["=", "!=", ">", ">=", "<", "<="],
            value=current.get("operator", ">="),
            layout=widgets.Layout(width="85px"),
        )
        right_col = widgets.Dropdown(
            options=[c["name"] for c in state["columns"]],
            value=current.get("right", state["columns"][1]["name"]),
            layout=widgets.Layout(width="220px"),
        )
        block = widgets.Checkbox(
            value=current.get("block", True),
            description="Block",
            indent=False,
        )
        save = widgets.Button(
            description="Save relationship",
            button_style="primary",
            layout=widgets.Layout(width="145px", height="34px"),
        )

        def save_clicked(_):
            state["relationships"][key] = [{
                "left": left_col.value,
                "operator": operator.value,
                "right": right_col.value,
                "block": block.value,
            }]
            refresh_relationship_selector()
            render_header()
            render_relationship_editor()
            status.value = "<span class='fops-success'>✓ Column comparison saved</span>"

        save.on_click(save_clicked)
        controls.extend([
            widgets.HTML("<div class='fops-field-title'>Comparison</div>"),
            widgets.HBox([left_col, operator, right_col], layout=widgets.Layout(gap="8px")),
            block,
            widgets.HBox([relationship_ai, save], layout=widgets.Layout(gap="8px")),
        ])

    elif key in {"required_when", "conditional_value"}:
        current = existing[0] if existing else {}
        condition_col = widgets.Dropdown(
            options=[c["name"] for c in state["columns"]],
            value=current.get("condition_column", "status"),
            layout=widgets.Layout(width="210px"),
        )
        operator = widgets.Dropdown(
            options=["=", "!=", ">", ">=", "<", "<="],
            value=current.get("operator", "="),
            layout=widgets.Layout(width="85px"),
        )
        condition_value = widgets.Text(
            value=current.get("condition_value", ""),
            placeholder="Condition value",
            layout=widgets.Layout(width="210px"),
        )
        target_col = widgets.Dropdown(
            options=[c["name"] for c in state["columns"]],
            value=current.get("target_column", state["columns"][0]["name"]),
            layout=widgets.Layout(width="220px"),
        )
        expected_value = widgets.Text(
            value=current.get("expected_value", ""),
            placeholder="Expected value",
            layout=widgets.Layout(width="220px"),
        )
        block = widgets.Checkbox(
            value=current.get("block", True),
            description="Block",
            indent=False,
        )
        save = widgets.Button(
            description="Save relationship",
            button_style="primary",
            layout=widgets.Layout(width="145px", height="34px"),
        )

        def save_clicked(_):
            record = {
                "condition_column": condition_col.value,
                "operator": operator.value,
                "condition_value": condition_value.value,
                "target_column": target_col.value,
                "block": block.value,
            }
            if key == "conditional_value":
                record["expected_value"] = expected_value.value
            state["relationships"][key] = [record]
            refresh_relationship_selector()
            render_header()
            render_relationship_editor()
            status.value = f"<span class='fops-success'>✓ {esc(meta['title'])} saved</span>"

        save.on_click(save_clicked)

        target_label = "Required column" if key == "required_when" else "Target column"
        controls.extend([
            widgets.HTML("<div class='fops-field-title'>When</div>"),
            widgets.HBox([condition_col, operator, condition_value], layout=widgets.Layout(gap="8px")),
            widgets.HTML(f"<div class='fops-field-title' style='margin-top:10px'>{target_label}</div>"),
            target_col,
        ])

        if key == "conditional_value":
            controls.extend([
                widgets.HTML("<div class='fops-field-title' style='margin-top:10px'>Expected value</div>"),
                expected_value,
            ])

        controls.extend([
            block,
            widgets.HBox([relationship_ai, save], layout=widgets.Layout(gap="8px")),
        ])

    else:
        current = existing[0] if existing else {}
        source_col = widgets.Dropdown(
            options=[c["name"] for c in state["columns"]],
            value=current.get("source_column", "product_id"),
            layout=widgets.Layout(width="220px"),
        )
        target_table = widgets.Text(
            value=current.get("target_table", ""),
            placeholder="products",
            layout=widgets.Layout(width="240px"),
        )
        target_col = widgets.Text(
            value=current.get("target_column", ""),
            placeholder="product_id",
            layout=widgets.Layout(width="220px"),
        )
        block = widgets.Checkbox(
            value=current.get("block", True),
            description="Block",
            indent=False,
        )
        save = widgets.Button(
            description="Save relationship",
            button_style="primary",
            layout=widgets.Layout(width="145px", height="34px"),
        )

        def save_clicked(_):
            state["relationships"][key] = [{
                "source_column": source_col.value,
                "target_table": target_table.value,
                "target_column": target_col.value,
                "block": block.value,
            }]
            refresh_relationship_selector()
            render_header()
            render_relationship_editor()
            status.value = "<span class='fops-success'>✓ Referential relationship saved</span>"

        save.on_click(save_clicked)
        controls.extend([
            widgets.HTML("<div class='fops-field-title'>Column</div>"),
            source_col,
            widgets.HTML("<div class='fops-field-title' style='margin-top:10px'>References</div>"),
            widgets.HBox([
                widgets.VBox([widgets.HTML("<div class='fops-note'>Table</div>"), target_table]),
                widgets.VBox([widgets.HTML("<div class='fops-note'>Column</div>"), target_col]),
            ], layout=widgets.Layout(gap="10px")),
            block,
            widgets.HBox([relationship_ai, save], layout=widgets.Layout(gap="8px")),
        ])

    relationship_editor.children = tuple(controls)


def render_relationships():
    refresh_relationship_selector()

    left.children = [
        widgets.HTML("""
            <div class='fops-section-label'>Relationship data quality</div>
            <div class='fops-note' style='margin-bottom:8px'>
                Multi column rules and rules that express a relationship between fields or tables.
            </div>
        """),
        relationship_selector,
    ]

    right.children = [relationship_editor]
    render_relationship_editor()


def relationship_changed(change):
    if change["name"] == "value" and change["new"]:
        selection["relationship"] = change["new"]
        render_relationship_editor()


def _manifest_yaml(value, indent=0):
    """Render a small dependency-free YAML view of the manifest."""
    pad = "  " * indent
    if isinstance(value, dict):
        lines = []
        for key, item in value.items():
            if isinstance(item, (dict, list)):
                lines.append(f"{pad}{key}:")
                lines.extend(_manifest_yaml(item, indent + 1))
            else:
                lines.append(f"{pad}{key}: {json.dumps(item, ensure_ascii=False)}")
        return lines
    if isinstance(value, list):
        lines = []
        for item in value:
            if isinstance(item, (dict, list)):
                lines.append(f"{pad}-")
                lines.extend(_manifest_yaml(item, indent + 1))
            else:
                lines.append(f"{pad}- {json.dumps(item, ensure_ascii=False)}")
        return lines
    return [f"{pad}{json.dumps(value, ensure_ascii=False)}"]


def build_data_contract_manifest():
    """Build the prototype manifest in the same top-level shape as FabricOps' frozen payload."""
    table = state["table"]
    contract = state["contract"]
    td = state["table_definition"]

    enrichment_table = []
    if td.get("description"):
        enrichment_table.append({
            "enrichment_level": "table",
            "enrichment_type": "Description",
            "value": td["description"],
        })
    if td.get("classification"):
        enrichment_table.append({
            "enrichment_level": "table",
            "enrichment_type": "Classification",
            "value": td["classification"],
        })

    enrichment_columns = []
    columns = []
    for index, column in enumerate(state["columns"], start=1):
        column_id = f"prototype-column-{index}"
        columns.append({
            "column_id": column_id,
            "column_name": column["name"],
            "data_type": column["type"],
        })
        if column.get("description"):
            enrichment_columns.append({
                "column_id": column_id,
                "enrichment_level": "column",
                "enrichment_type": "Description",
                "value": column["description"],
            })
        if column.get("classification"):
            enrichment_columns.append({
                "column_id": column_id,
                "enrichment_level": "column",
                "enrichment_type": "Classification",
                "value": column["classification"],
            })

    guardrails = []

    required_columns = [c["name"] for c in state["columns"] if c.get("required")]
    if required_columns:
        guardrails.append({
            "guardrail_type": "schema",
            "rule_type": "required_columns",
            "action": "Block",
            "rule_parameters": {"required_columns": required_columns},
        })

    if td.get("freshness_enabled"):
        guardrails.append({
            "guardrail_type": "freshness",
            "rule_type": "maximum_age",
            "action": "Block" if td.get("freshness_block") else "Warn",
            "rule_parameters": {
                "column": td.get("freshness_column"),
                "maximum_age": td.get("freshness_age"),
                "unit": str(td.get("freshness_unit") or "").lower(),
            },
        })

    if td.get("source_drift_enabled"):
        guardrails.append({
            "guardrail_type": "source_drift",
            "rule_type": "source_drift",
            "action": "Block" if td.get("source_drift_block") else "Warn",
            "rule_parameters": {
                "partition": td.get("source_drift_partition") or None,
                "change_column": td.get("source_drift_change"),
            },
        })

    for index, column in enumerate(state["columns"], start=1):
        column_id = f"prototype-column-{index}"

        if column.get("sensitive_enabled"):
            guardrails.append({
                "column_id": column_id,
                "guardrail_type": "sensitive_data",
                "rule_type": "sensitive_data",
                "action": "Block" if column.get("sensitive_block") else "Warn",
                "rule_parameters": {
                    "scope": "column",
                    "treatment": column.get("treatment") or "tokenize",
                },
            })

        for key, config in column.get("dq", {}).items():
            if not config.get("enabled", True):
                continue
            params = {
                k: v for k, v in config.items()
                if k not in {"enabled", "block"}
            }
            params["column"] = column["name"]
            guardrails.append({
                "column_id": column_id,
                "guardrail_type": "data_quality",
                "rule_type": key,
                "action": "Block" if config.get("block", True) else "Warn",
                "rule_parameters": params,
            })

    # Advanced rules are still Data Quality Guardrails in the canonical payload.
    for key, records in state["relationships"].items():
        for record in records:
            guardrails.append({
                "guardrail_type": "data_quality",
                "rule_type": key,
                "action": "Block" if record.get("block", True) else "Warn",
                "rule_parameters": {
                    k: v for k, v in record.items()
                    if k != "block"
                },
            })

    return {
        "contract": {
            "contract_id": "prototype-contract",
            "contract_version": int(contract["version"]),
            "status": str(contract["status"]).lower(),
        },
        "table": {
            "table_id": table.get("table_id", "prototype-table"),
            "environment_name": table.get("environment"),
            "store_type": table.get("store_type"),
            "layer": table.get("store"),
            "schema_name": table.get("schema"),
            "table_name": table.get("table_name"),
            "columns": columns,
            "processing": {
                "load_strategy": table.get("load_strategy"),
            },
            "writer": {
                "notebook_id": None,
                "notebook_name": None,
            },
        },
        "enrichment": {
            "table": enrichment_table,
            "columns": enrichment_columns,
        },
        "guardrails": guardrails,
    }



EXAMPLE_COLUMN_VALUES = {
    "order_id": "ORD-1001",
    "customer_id": "CUST-001",
    "product_id": "PROD-101",
    "order_datetime": "2026-09-22 20:10:00",
    "quantity": "2",
    "unit_price": "49.90",
    "order_net_amount": "99.80",
    "historical_net_amount": "89.80",
    "status": "Completed",
}


def render_review():
    """Render the canonical contract manifest for human and AI review before freeze."""
    global DATA_CONTRACT_MANIFEST, DATA_CONTRACT_MANIFEST_JSON

    table = state["table"]
    contract = state["contract"]
    manifest = build_data_contract_manifest()

    DATA_CONTRACT_MANIFEST = manifest
    DATA_CONTRACT_MANIFEST_JSON = json.dumps(
        manifest, indent=2, ensure_ascii=False, default=str
    )
    guardrail_counts = {}
    for rule in manifest["guardrails"]:
        key = str(rule.get("guardrail_type") or "unknown").replace("_", " ").title()
        guardrail_counts[key] = guardrail_counts.get(key, 0) + 1

    guardrail_rows = "".join(
        f"""
        <div style='display:flex;justify-content:space-between;gap:16px;padding:5px 0'>
            <span>{esc(name)}</span><b>{count}</b>
        </div>
        """
        for name, count in sorted(guardrail_counts.items())
    ) or "<div class='fops-left-muted'>None configured</div>"

    processing = manifest["table"]["processing"]

    required_columns = set()
    for rule in manifest["guardrails"]:
        if (
            rule.get("guardrail_type") == "schema"
            and rule.get("rule_type") == "required_columns"
        ):
            required_columns.update(
                (rule.get("rule_parameters") or {}).get("required_columns") or []
            )

    table_enrichment = {
        row["enrichment_type"]: row["value"]
        for row in manifest["enrichment"]["table"]
    }

    human_guardrails = []
    for rule in manifest["guardrails"]:
        kind = str(rule.get("guardrail_type") or "").replace("_", " ").title()
        rule_type = str(rule.get("rule_type") or "").replace("_", " ")
        action = rule.get("action") or "Warn"
        params = rule.get("rule_parameters") or {}
        detail = ", ".join(
            f"{k.replace('_', ' ')}: {v}"
            for k, v in params.items()
            if v not in (None, "", [], {})
        )
        human_guardrails.append(
            f"""
            <div style='padding:7px 0;border-bottom:1px solid #eef1f4'>
                <div style='display:flex;justify-content:space-between;gap:16px'>
                    <b>{esc(kind)} · {esc(rule_type)}</b>
                    <span>{esc(action)}</span>
                </div>
                <div class='fops-left-muted' style='margin-top:3px'>{esc(detail or "No parameters")}</div>
            </div>
            """
        )
    human_guardrails_html = "".join(human_guardrails) or "<div class='fops-left-muted'>None configured</div>"

    column_rows = "".join(
        f"""
        <div style='display:grid;
                    grid-template-columns:minmax(160px,1.05fr) 120px 85px minmax(150px,1fr) minmax(220px,1.5fr);
                    gap:12px;padding:6px 0;border-bottom:1px solid #eef1f4;align-items:start'>
            <span><b>{esc(c["column_name"])}</b></span>
            <span>{esc(c["data_type"])}</span>
            <span>{"Yes" if c["column_name"] in required_columns else "No"}</span>
            <span class='fops-left-muted'>{esc(EXAMPLE_COLUMN_VALUES.get(c["column_name"], "—"))}</span>
            <span class='fops-left-muted'>{esc(next((
                row["value"] for row in manifest["enrichment"]["columns"]
                if row.get("column_id") == c["column_id"]
                and row.get("enrichment_type") == "Description"
            ), ""))}</span>
        </div>
        """
        for c in manifest["table"]["columns"]
    )

    left.children = [
        widgets.HTML(
            f"""
            <div class='fops-section-label'>Manifest</div>
            <div class='fops-focus-title' style='font-size:18px'>
                {esc(table["schema"])}.{esc(table["table_name"])}
            </div>
            <div class='fops-left-muted' style='margin-top:5px'>
                v{contract["version"]} · {esc(contract["status"])}
            </div>

            <div class='fops-divider'></div>

            <div class='fops-left-label'>Sections</div>
            <div class='fops-left-muted' style='line-height:1.9;margin-top:5px'>
                Identity<br>
                Table & processing<br>
                Columns<br>
                Guardrails & advanced rules<br>
                JSON manifest
            </div>
            """
        ),
        widgets.HTML(
            """
            <div class='fops-note' style='margin-top:12px'>
                Notebook variable<br>
                <b>DATA_CONTRACT_MANIFEST</b>
            </div>
            """
        ),
    ]

    json_escaped = esc(DATA_CONTRACT_MANIFEST_JSON)
    right.children = [
        widgets.HTML(
            f"""
            <div class='fops-title-bar'>
                <div class='fops-field-title'>Contract manifest</div>
            </div>

            <div style='padding:10px 4px 16px 4px'>
                <div class='fops-left-label'>Identity</div>
                <div class='fops-left-value'>
                    {esc(table["schema"])}.{esc(table["table_name"])}
                    · v{contract["version"]} · {esc(contract["status"])}
                </div>
            </div>

            <div class='fops-title-bar'>
                <div class='fops-field-title'>Table & processing</div>
            </div>
            <div style='padding:10px 4px 16px 4px'>
                <div><b>Description:</b> {esc(table_enrichment.get("Description") or "Not provided")}</div>
                <div style='margin-top:5px'><b>Classification:</b> {esc(table_enrichment.get("Classification") or "Not classified")}</div>
                <div style='margin-top:5px'><b>Load strategy:</b> {esc(processing.get("load_strategy") or "Not defined")}</div>
            </div>

            <div class='fops-title-bar'>
                <div class='fops-field-title'>Columns</div>
            </div>
            <div style='padding:8px 4px 16px 4px'>
                <div style='display:grid;
                            grid-template-columns:minmax(160px,1.05fr) 120px 85px minmax(150px,1fr) minmax(220px,1.5fr);
                            gap:12px;padding:4px 0 6px 0;border-bottom:1px solid #dfe5eb'>
                    <b>Column</b>
                    <b>Datatype</b>
                    <b>Required</b>
                    <b>Example value</b>
                    <b>Description</b>
                </div>
                {column_rows}
                <div class='fops-note' style='margin-top:8px'>
                    Example values are prototype display samples only and are not stored in the frozen contract manifest.
                </div>
            </div>

            <div class='fops-title-bar'>
                <div class='fops-field-title'>Guardrails & advanced rules</div>
            </div>
            <div style='padding:8px 4px 16px 4px'>
                {human_guardrails_html}
            </div>

            <details style='margin-top:12px'>
                <summary style='cursor:pointer;font-weight:600'>JSON manifest</summary>
                <pre style='white-space:pre-wrap;word-break:break-word;background:#f7f8fa;padding:12px;border:1px solid #e1e6eb'>{json_escaped}</pre>
            </details>

            """
        )
    ]

    if str(contract["status"]).upper() == "DRAFT":
        freeze_button = widgets.Button(
            description=f"Freeze v{contract['version']}",
            button_style="primary",
            layout=widgets.Layout(width="130px", height="34px"),
        )

        def freeze_clicked(_):
            state["contract"]["status"] = "FROZEN"
            status.value = "<span class='fops-success'>✓ Contract manifest frozen</span>"
            render_review()

        freeze_button.on_click(freeze_clicked)
        right.children = tuple(right.children) + (
            widgets.HBox([freeze_button], layout=widgets.Layout(justify_content="flex-end")),
        )


relationship_selector.observe(relationship_changed, names="value")


In [ ]:
relationship_selector = widgets.Select(
    rows=5,
    layout=widgets.Layout(width="100%", height="125px"),
)

relationship_instance_selector = widgets.Select(
    rows=8,
    layout=widgets.Layout(width="100%", height="235px"),
)

relationship_editor = widgets.VBox(
    layout=widgets.Layout(
        width="100%",
        align_items="stretch",
        gap="10px",
        padding="10px 12px",
    )
)

relationship_ai = suggest_button("✨ Suggest rules")

new_relationship_config_button = widgets.Button(
    description="+ New configuration",
    layout=widgets.Layout(width="150px", height="32px"),
)


def refresh_relationship_selector():
    relationship_selector.options = [
        (
            f"{meta['title']}   ·   {len(state['relationships'].get(key, []))} configured",
            key,
        )
        for key, meta in RELATIONSHIP_RULES.items()
    ]
    relationship_selector.value = selection["relationship"]


def relationship_summary(key, record):
    if key == "composite_unique":
        return " + ".join(record.get("columns", [])) or "New combination"

    if key == "compare_columns":
        return (
            f"{record.get('left', 'Column A')} "
            f"{record.get('operator', '=')} "
            f"{record.get('right', 'Column B')}"
        )

    if key == "required_when":
        return (
            f"{record.get('condition_column', 'condition')} "
            f"{record.get('operator', '=')} "
            f"{record.get('condition_value', 'value')} → "
            f"{record.get('target_column', 'target')} required"
        )

    if key == "conditional_value":
        return (
            f"{record.get('condition_column', 'condition')} "
            f"{record.get('operator', '=')} "
            f"{record.get('condition_value', 'value')} → "
            f"{record.get('target_column', 'target')} = "
            f"{record.get('expected_value', 'value')}"
        )

    return (
        f"{record.get('source_column', 'column')} → "
        f"{record.get('target_table', 'table')}."
        f"{record.get('target_column', 'column')}"
    )


def refresh_relationship_instances():
    key = selection["relationship"]
    records = state["relationships"].get(key, [])

    options = [
        (f"{i + 1}. {relationship_summary(key, record)}", i)
        for i, record in enumerate(records)
    ]

    relationship_instance_selector.options = options

    idx = selection.get("relationship_index")
    if idx is not None and idx >= len(records):
        idx = None
        selection["relationship_index"] = None

    values = [value for _, value in options]
    relationship_instance_selector.value = idx if idx in values else None


def render_relationship_editor():
    key = selection["relationship"]
    meta = RELATIONSHIP_RULES[key]
    records = state["relationships"].get(key, [])
    idx = selection.get("relationship_index")
    editing = idx is not None and 0 <= idx < len(records)
    current = records[idx] if editing else {}

    controls = [
        widgets.HTML(
            f"<div class='fops-left-label'>"
            f"{'Editing configuration ' + str(idx + 1) if editing else 'New configuration'}"
            f"</div>"
        )
    ]

    if key == "composite_unique":
        checks = [
            widgets.Checkbox(
                value=c["name"] in current.get("columns", []),
                description=f"{c['name']} · {c['type']}",
                indent=False,
                layout=widgets.Layout(width="100%"),
            )
            for c in state["columns"]
        ]

        block = widgets.Checkbox(
            value=current.get("block", True),
            description="Block on failure",
            indent=False,
            layout=widgets.Layout(width="150px"),
        )

        controls.extend([
            widgets.HTML("<div class='fops-subheading'>Columns in this combination</div>"),
            widgets.VBox(checks, layout=widgets.Layout(width="100%", gap="3px")),
            widgets.VBox(
                [
                    widgets.HTML("<div class='fops-field-title'>Enforcement</div>"),
                    block,
                ],
                layout=widgets.Layout(
                    width="220px",
                    max_width="100%",
                    gap="4px",
                    margin="10px 0 0 0",
                ),
            ),
        ])

        def build_record():
            cols = [c["name"] for c, cb in zip(state["columns"], checks) if cb.value]
            if len(cols) < 2:
                raise ValueError("Select at least two columns.")
            return {"columns": cols, "block": block.value}

    elif key == "compare_columns":
        names = [c["name"] for c in state["columns"]]
        left_col = widgets.Dropdown(
            options=names,
            value=current.get("left", names[0]),
            layout=widgets.Layout(width="240px"),
        )
        operator = widgets.Dropdown(
            options=["=", "!=", ">", ">=", "<", "<="],
            value=current.get("operator", ">="),
            layout=widgets.Layout(width="90px"),
        )
        right_col = widgets.Dropdown(
            options=names,
            value=current.get("right", names[1]),
            layout=widgets.Layout(width="240px"),
        )
        block = widgets.Checkbox(
            value=current.get("block", True),
            description="Block on failure",
            indent=False,
            layout=widgets.Layout(width="150px"),
        )

        controls.extend([
            widgets.HTML("<div class='fops-subheading'>Comparison</div>"),
            widgets.HBox([left_col, operator, right_col], layout=widgets.Layout(gap="10px")),
            widgets.VBox(
                [
                    widgets.HTML("<div class='fops-field-title'>Enforcement</div>"),
                    block,
                ],
                layout=widgets.Layout(
                    width="220px",
                    max_width="100%",
                    gap="4px",
                    margin="10px 0 0 0",
                ),
            ),
        ])

        def build_record():
            return {
                "left": left_col.value,
                "operator": operator.value,
                "right": right_col.value,
                "block": block.value,
            }

    elif key in {"required_when", "conditional_value"}:
        names = [c["name"] for c in state["columns"]]
        condition_col = widgets.Dropdown(
            options=names,
            value=current.get("condition_column", "status" if "status" in names else names[0]),
            layout=widgets.Layout(width="225px"),
        )
        operator = widgets.Dropdown(
            options=["=", "!=", ">", ">=", "<", "<="],
            value=current.get("operator", "="),
            layout=widgets.Layout(width="90px"),
        )
        condition_value = widgets.Text(
            value=current.get("condition_value", ""),
            placeholder="Condition value",
            layout=widgets.Layout(width="225px"),
        )
        target_col = widgets.Dropdown(
            options=names,
            value=current.get("target_column", names[0]),
            layout=widgets.Layout(width="240px"),
        )
        expected_value = widgets.Text(
            value=current.get("expected_value", ""),
            placeholder="Expected value",
            layout=widgets.Layout(width="240px"),
        )
        block = widgets.Checkbox(
            value=current.get("block", True),
            description="Block on failure",
            indent=False,
            layout=widgets.Layout(width="150px"),
        )

        controls.extend([
            widgets.HTML("<div class='fops-subheading'>When</div>"),
            widgets.VBox([
                widgets.HTML("<div class='fops-note'>Condition column</div>"),
                condition_col,
            ], layout=widgets.Layout(width="280px", max_width="100%", gap="4px")),
            widgets.VBox([
                widgets.HTML("<div class='fops-note'>Operator</div>"),
                operator,
            ], layout=widgets.Layout(width="130px", max_width="100%", gap="4px")),
            widgets.VBox([
                widgets.HTML("<div class='fops-note'>Condition value</div>"),
                condition_value,
            ], layout=widgets.Layout(width="280px", max_width="100%", gap="4px")),
            widgets.VBox(
                [
                    widgets.HTML(
                        f"<div class='fops-subheading'>"
                        f"{'Required column' if key == 'required_when' else 'Target column'}"
                        f"</div>"
                    ),
                    target_col,
                ],
                layout=widgets.Layout(width="280px", max_width="100%", gap="4px"),
            ),
        ])

        if key == "conditional_value":
            controls.extend([
                widgets.VBox(
                    [
                        widgets.HTML("<div class='fops-subheading'>Expected value</div>"),
                        expected_value,
                    ],
                    layout=widgets.Layout(width="280px", max_width="100%", gap="4px"),
                ),
            ])

        controls.extend([
            widgets.VBox(
                [
                    widgets.HTML("<div class='fops-field-title'>Enforcement</div>"),
                    block,
                ],
                layout=widgets.Layout(
                    width="220px",
                    max_width="100%",
                    gap="4px",
                    margin="10px 0 0 0",
                ),
            ),
        ])

        def build_record():
            record = {
                "condition_column": condition_col.value,
                "operator": operator.value,
                "condition_value": condition_value.value,
                "target_column": target_col.value,
                "block": block.value,
            }
            if key == "conditional_value":
                record["expected_value"] = expected_value.value
            return record

    else:
        names = [c["name"] for c in state["columns"]]
        source_col = widgets.Dropdown(
            options=names,
            value=current.get("source_column", "product_id" if "product_id" in names else names[0]),
            layout=widgets.Layout(width="240px"),
        )
        target_table = widgets.Text(
            value=current.get("target_table", ""),
            placeholder="products",
            layout=widgets.Layout(width="250px"),
        )
        target_col = widgets.Text(
            value=current.get("target_column", ""),
            placeholder="product_id",
            layout=widgets.Layout(width="240px"),
        )
        block = widgets.Checkbox(
            value=current.get("block", True),
            description="Block on failure",
            indent=False,
            layout=widgets.Layout(width="150px"),
        )

        controls.extend([
            widgets.VBox(
                [
                    widgets.HTML("<div class='fops-subheading'>Source column</div>"),
                    source_col,
                ],
                layout=widgets.Layout(width="280px", max_width="100%", gap="4px"),
            ),
            widgets.HTML("<div class='fops-subheading'>References</div>"),
            widgets.VBox([
                widgets.HTML("<div class='fops-note'>Table</div>"),
                target_table,
            ], layout=widgets.Layout(width="300px", max_width="100%", gap="4px")),
            widgets.VBox([
                widgets.HTML("<div class='fops-note'>Column</div>"),
                target_col,
            ], layout=widgets.Layout(width="280px", max_width="100%", gap="4px")),
            widgets.VBox(
                [
                    widgets.HTML("<div class='fops-field-title'>Enforcement</div>"),
                    block,
                ],
                layout=widgets.Layout(
                    width="220px",
                    max_width="100%",
                    gap="4px",
                    margin="10px 0 0 0",
                ),
            ),
        ])

        def build_record():
            return {
                "source_column": source_col.value,
                "target_table": target_table.value,
                "target_column": target_col.value,
                "block": block.value,
            }

    save = widgets.Button(
        description="Save configuration",
        button_style="primary",
        layout=widgets.Layout(width="160px", height="34px"),
    )

    remove = widgets.Button(
        description="Remove",
        layout=widgets.Layout(width="95px", height="34px"),
        disabled=not editing,
    )

    def save_clicked(_):
        try:
            record = build_record()
        except ValueError as exc:
            status.value = f"<span style='color:#b42318'>{esc(exc)}</span>"
            return

        if editing:
            state["relationships"][key][idx] = record
            message = f"{meta['title']} configuration updated"
        else:
            state["relationships"][key].append(record)
            selection["relationship_index"] = len(state["relationships"][key]) - 1
            message = f"{meta['title']} configuration added"

        refresh_relationship_selector()
        refresh_relationship_instances()
        render_header()
        render_relationship_editor()
        status.value = f"<span class='fops-success'>✓ {esc(message)}</span>"

    def remove_clicked(_):
        if not editing:
            return
        state["relationships"][key].pop(idx)
        selection["relationship_index"] = None
        refresh_relationship_selector()
        refresh_relationship_instances()
        render_header()
        render_relationship_editor()
        status.value = f"<span class='fops-success'>✓ {esc(meta['title'])} configuration removed</span>"

    save.on_click(save_clicked)
    remove.on_click(remove_clicked)

    action_controls = [save]
    if editing:
        action_controls.append(remove)

    controls.extend([
        widgets.HTML("<div class='fops-divider'></div>"),
        widgets.HBox(
            action_controls,
            layout=widgets.Layout(
                gap="8px",
                justify_content="flex-start",
                width="100%",
            ),
        ),
    ])

    relationship_editor.children = tuple(controls)


def render_relationships():
    refresh_relationship_selector()
    refresh_relationship_instances()

    left.children = [
        widgets.HTML("<div class='fops-section-label'>Advanced rules</div>"),
        relationship_ai,
        widgets.HTML("<div class='fops-left-label' style='margin-top:12px'>Rule type</div>"),
        relationship_selector,
        widgets.HTML(
            "<div class='fops-left-label' style='margin-top:14px'>Saved configurations</div>"
        ),
        relationship_instance_selector,
        new_relationship_config_button,
    ]

    right.children = [relationship_editor]
    render_relationship_editor()


def relationship_changed(change):
    if change["name"] == "value" and change["new"]:
        selection["relationship"] = change["new"]
        selection["relationship_index"] = None
        refresh_relationship_instances()
        render_relationship_editor()


def relationship_instance_changed(change):
    if change["name"] == "value":
        selection["relationship_index"] = change["new"]
        render_relationship_editor()


def new_relationship_configuration(_):
    selection["relationship_index"] = None
    relationship_instance_selector.value = None
    render_relationship_editor()
    status.value = ""


relationship_selector.observe(relationship_changed, names="value")
relationship_instance_selector.observe(relationship_instance_changed, names="value")
new_relationship_config_button.on_click(new_relationship_configuration)


In [ ]:
def render_current_view():
    status.value = ""

    if top_nav.value == "table":
        render_table()

    elif top_nav.value == "columns":
        refresh_column_selector()
        render_column(selection["column"])

    elif top_nav.value == "relationships":
        render_relationships()

    else:
        render_review()


def top_nav_changed(change):
    if change["name"] == "value":
        render_current_view()


top_nav.observe(top_nav_changed, names="value")


render_table()

# Start at table selection. The contract editor appears only after Load Table.
editor_shell.layout.display = "none"

app = widgets.VBox(
    [
        css,
        selector_panel,
        editor_shell,
    ],
    layout=widgets.Layout(width="100%", gap="8px"),
)

app.add_class("fops-root")

display(app)
